In [11]:
import pandas as pd

meta = pd.read_csv("data/metadata.csv")
print(meta.shape)
print(meta.columns.tolist())
print(meta.head())
print(meta["type"].value_counts())
print("電池の数:", meta["battery_id"].nunique())

(7565, 10)
['type', 'start_time', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct']
        type                                         start_time  \
0  discharge  [2010.       7.      21.      15.       0.    ...   
1  impedance  [2010.       7.      21.      16.      53.    ...   
2     charge  [2010.       7.      21.      17.      25.    ...   
3  impedance                    [2010    7   21   20   31    5]   
4  discharge  [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...   

   ambient_temperature battery_id  test_id  uid   filename  \
0                    4      B0047        0    1  00001.csv   
1                   24      B0047        1    2  00002.csv   
2                    4      B0047        2    3  00003.csv   
3                   24      B0047        3    4  00004.csv   
4                    4      B0047        4    5  00005.csv   

             Capacity                   Re                  Rct  
0  1.6743047446975208           

In [12]:
import re #正規表現

def parse_time(s):
    nums = [float(x) for x in re.findall(r"[-+]?\d+\.?\d*(?:[eE][-+]?\d+)?", s)]
    y, mo, d, h, mi, sec = nums[:6]
    return pd.Timestamp(int(y), int(mo), int(d), int(h), int(mi)) + pd.to_timedelta(sec, unit="s")

meta["time"] = meta["start_time"].apply(parse_time)
for c in ["Capacity", "Re", "Rct"]:
    meta[c] = pd.to_numeric(meta[c], errors="coerce")

print(meta[["time", "Capacity", "Re", "Rct"]].describe())
print(meta.groupby("battery_id")["type"].value_counts().unstack())
print(meta.groupby("type")["ambient_temperature"].value_counts())

                                time     Capacity            Re           Rct
count                           7565  2769.000000  1.947000e+03  1.947000e+03
mean   2009-08-14 03:15:31.312608768     1.326543 -4.976500e+11  1.055903e+12
min       2008-04-02 13:08:17.921000     0.000000 -9.689245e+14 -2.091081e+02
25%    2008-07-11 18:25:27.592999936     1.150286  5.782157e-02  8.155754e-02
50%    2009-08-02 09:10:15.046000128     1.428065  7.255344e-02  1.014191e-01
75%    2010-06-30 09:21:28.343000064     1.673645  9.229960e-02  1.565123e-01
max       2010-09-30 15:32:33.078000     2.640149  4.482291e+02  2.055843e+15
std                              NaN     0.472517  2.195872e+13  4.659154e+13
type        charge  discharge  impedance
battery_id                              
B0005          170        168        278
B0006          170        168        278
B0007          170        168        278
B0018          134        132         53
B0025           31         28         21
B0026      

In [13]:
imp = meta[meta["type"] == "impedance"]
print("インピーダンス:", len(imp),
      " Re異常:", (~imp["Re"].between(0, 1)).sum(),
      " Rct異常:", (~imp["Rct"].between(0, 1)).sum())

dis = meta[meta["type"] == "discharge"].sort_values("time")
print("容量0以下:", (dis["Capacity"] <= 0).sum(), " 2Ah超:", (dis["Capacity"] > 2).sum())

summary = dis.groupby("battery_id").agg(
    放電温度=("ambient_temperature", lambda s: sorted(s.unique())),
    最初の容量=("Capacity", "first"),
    最小容量=("Capacity", "min"),
    件数=("Capacity", "size"),
)
print(summary)


imp = meta[meta["type"] == "impedance"]
print("インピーダンス:", len(imp),
      "Re異常:", len(imp))

インピーダンス: 1956  Re異常: 23  Rct異常: 23
容量0以下: 19  2Ah超: 11
                放電温度     最初の容量      最小容量   件数
battery_id                                   
B0005           [24]  1.856487  1.287453  168
B0006           [24]  2.035338  1.153818  168
B0007           [24]  1.891052  1.400455  168
B0018           [24]  1.855005  1.341051  132
B0025           [24]  1.847011  1.767789   28
B0026           [24]  1.813250  1.386337   28
B0027           [24]  1.823308  1.770093   28
B0028           [24]  1.804685  1.717234   28
B0029           [43]  1.697507  1.612080   40
B0030           [43]  1.656071  1.562780   40
B0031           [43]  1.666675  1.666675   40
B0032           [43]  1.704864  1.635800   40
B0033           [24]  0.068426  0.068426  197
B0034           [24]  0.745930  0.745930  197
B0036           [24]  1.001983  1.001983  197
B0038       [24, 44]  0.898057  0.898057   47
B0039       [24, 44]  0.119038  0.119038   47
B0040       [24, 44]  0.673463  0.425812   47
B0041            [4]  0.0

In [14]:
CORE = ["B0005", "B0006", "B0007", "B0018", "B0025", "B0026", "B0027", "B0028"]
m = meta[meta["battery_id"].isin(CORE)]

# インピーダンス測定（おかしな値は除く）
imp = m[(m["type"] == "impedance") & m["Re"].between(0, 1) & m["Rct"].between(0, 1)]
imp = imp[["battery_id", "time", "Re", "Rct"]].sort_values("time")

# 放電（容量0などは除く）
dis = m[(m["type"] == "discharge") & m["Capacity"].between(0.1, 2.5)]
dis = dis[["battery_id", "time", "Capacity"]].rename(columns={"time": "dis_time"}).sort_values("dis_time")

# 各インピーダンス測定に「同じ電池の、直前の放電」をくっつける
df = pd.merge_asof(imp, dis, left_on="time", right_on="dis_time",
                   by="battery_id", direction="backward")

df["gap_h"] = (df["time"] - df["dis_time"]).dt.total_seconds() / 3600
df["SOH"] = df["Capacity"] / 2.0
df["劣化"] = (df["SOH"] < 0.8).astype(int)

print("件数:", len(df), " 容量がつかなかった:", df["Capacity"].isna().sum())
print(df["gap_h"].describe())
print(df.groupby("battery_id").agg(件数=("SOH", "size"), 劣化=("劣化", "sum"), SOH最小=("SOH", "min")))

件数: 971  容量がつかなかった: 9
count    962.000000
mean       4.154990
std       17.791672
min        0.804991
25%        0.881675
50%        1.880352
75%        4.387114
max      310.148438
Name: gap_h, dtype: float64
             件数   劣化     SOH最小
battery_id                    
B0005       278  173  0.643726
B0006       278  196  0.576909
B0007       278  147  0.700228
B0018        53   26  0.673115
B0025        21    0  0.883895
B0026        21    2  0.693169
B0027        21    0  0.885046
B0028        21    0  0.858617


In [15]:
# ① 間の時間が長い組がどれくらいあるか見る
for h in [6, 12, 24, 48]:
    print(f"{h}時間超:", (df["gap_h"] > h).sum(), "件")

6時間超: 22 件
12時間超: 18 件
24時間超: 14 件
48時間超: 7 件


In [18]:
# ② 容量がつかなかった行と、間が長すぎる行を捨てる
LIMIT = 12    # ①を見て、自分で決める（単位は時間）

df2 = df.dropna(subset=["Capacity"])          # 容量が空の行を捨てる
df2 = df2[df2["gap_h"] <= 12]           # 間が LIMIT 以下の行だけ残す

print("残った件数:", len(df2))
print("劣化:", df2["劣化"].sum(), " 正常:", (df2["劣化"] == 0).sum())

残った件数: 944
劣化: 538  正常: 406
